# STEP00. 눈 트랙 전체 개요

## 이 노트북의 역할

눈 감김 파이프라인(MRL + DMD → 눈 상태 CNN → 검증)의 **지도**다. 어떤 노트북을 어떤 순서로 돌리는지, 각 STEP 이 무엇을 입력받아 무엇을 내는지, 지금까지의 결과가 무엇인지 한 곳에 모았다. 계산은 하지 않고 산출물 존재만 점검한다.

## 프로젝트 한 줄 요약

웹캠·주행 영상에서 운전자의 **눈 감김**을 CNN 으로 분류하고, 시간 누적 지표(PERCLOS·최장 감김)로 졸음 위험을 판정하는 파이프라인. 이 저장소의 눈 트랙은 그중 **눈 상태 분류기와 그 검증**을 담당한다. 하품 트랙과 위험도 지수 통합은 별도(팀원·후속).

## 데이터셋과 역할

| 데이터 | 규모 | 성격 | 역할 |
|---|---|---|---|
| MRL Eye | 84,898장 / 37명 | IR 그레이스케일 눈 crop | 눈 CNN **주 학습 데이터** |
| DMD | 16영상 / 13명 | RGB 주행 자세, `Car Stopped` | 도메인 보강 학습 + hold-out 평가 |
| Yawn-Eye (dataset_new) | Kaggle | 얼굴/눈 이미지 | 01·02 baseline CNN 학습용 |
| NITYMED | 126영상 | microsleep 19 / 하품 107 | 실제 졸음 라벨 검증 (STEP14·15) |

## 노트북 구성

번호의 십의 자리가 트랙을 나눈다: 0 공통 · 1 기준본 · 10번대 눈(이 작업) · 20번대 하품(팀원) · 30번대 통합.

| 번호 | 위치 | 역할 | 상태 |
|---|---|---|---|
| 00 | `notebooks/` | 이 지도 | — |
| 01_baseline_nithishm | `notebooks/` | NITHISH 원본 CNN (참고) | — |
| 01_TRAIN | `model/` | Eye/Yawn CNN 학습 (Yawn-Eye) | **수정 금지** |
| 02_INFER_YuNet | `model/` | YuNet 검출 + 추론 + 실시간 PERCLOS | **수정 금지** |
| 10_eye_gt_dmd | `notebooks/` | DMD annotation → 프레임 눈 상태 GT | 눈 트랙 |
| 11_eye_dataset_build | `notebooks/` | MRL + DMD → 통합 manifest | 눈 트랙 |
| 12_eye_train_eval | `notebooks/` | 눈 CNN 학습 A/B/C/C' 비교 | 눈 트랙 |
| 13_eye_frame_eval_dmd | `notebooks/` | DMD 영상 프레임 파이프라인 평가 | 눈 트랙 |
| 14_eye_nitymed_video_eval | `notebooks/` | NITYMED 영상 단위 판별 | 눈 트랙 |
| 15_eye_yawn_falsepositive | `notebooks/` | 하품 오경보 분석 | 눈 트랙 |
| 19_eye_train_colab | `notebooks/` | Colab GPU 학습판 | — |
| 20·21 | — | 하품 GT / 하품 CNN | 번호 예약 (팀원) |
| 30 | — | 위험도 지수 통합 | 번호 예약 |

## 실행 순서

```
[사전] model/01_TRAIN 실행 → eye_model.keras (또는 Release 에서 받기)
       src/mrl_split.py 실행 → outputs/mrl_split/mrl_manifest_subject.csv

10  DMD annotation → 프레임 GT (outputs/dmd_gt/)
     ↓
11  MRL + DMD → 통합 manifest (outputs/eye_dataset/eye_manifest.csv)
     ↓  (내부에서 src/build_dmd_eye_dataset.py 로 DMD 눈 crop 생성)
12  눈 CNN 학습 A/B/C/C' → model/artifacts/eye_*.keras
     ↓
13  DMD 영상 프레임 평가        14  NITYMED 영상 판별        15  하품 오경보
     (12 의 C 모델을 공유)
```

12 이후 13·14·15 는 서로 독립이며 모두 12 의 C 모델(`eye_mrl+dmd__eval-dmd__gray128.keras`)과 임계값 0.93 을 쓴다.

## src/ 모듈

| 모듈 | 역할 | 쓰는 노트북 |
|---|---|---|
| `dmd_annotation.py` | DMD OpenLABEL 파싱 | 10 |
| `dmd_crop.py` | mosaic 좌하 얼굴 타일 크롭 | 10·11 |
| `build_dmd_eye_dataset.py` | DMD 눈 crop 생성 + 경로/YuNet 헬퍼 | 10·11·13·14·15 |
| `mrl_split.py` | MRL subject 단위 분할 manifest | 11 (사전) |
| `mrl_dataset.py` · `eye_preprocess.py` | manifest → tf.data, 공용 전처리 | 12 |
| `train_eye.py` | 통합 manifest 로 A/B/C/C' 학습·평가 | 12 |
| `train_eye_mrl.py` | (구) MRL 전용 학습 | 19 |

In [ ]:
# [셀 1] 환경·산출물 점검 (지도 노트북이라 계산은 최소)

import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())

_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

print("PROJECT_ROOT :", config.PROJECT_ROOT.name)
print()

# 각 STEP 의 핵심 산출물이 있는지만 확인한다. 없으면 그 STEP 을 아직 안 돌린 것.
checks = [
    ("STEP10 DMD 프레임 GT",   config.OUTPUTS_DIR / "dmd_gt" / "_summary.csv"),
    ("STEP11 통합 manifest",   config.OUTPUTS_DIR / "eye_dataset" / "eye_manifest.csv"),
    ("STEP12 조건 비교",       config.OUTPUTS_DIR / "eye_dataset" / "step12_comparison.csv"),
    ("STEP12 C 모델",          config.ARTIFACT_DIR / "eye_mrl+dmd__eval-dmd__gray128.keras"),
    ("STEP13 프레임 평가",     config.OUTPUTS_DIR / "eye_frame_eval" / "frame_eval_summary.csv"),
    ("STEP14 NITYMED 영상평가", config.OUTPUTS_DIR / "nitymed_eval" / "video_metrics.csv"),
    ("STEP15 하품 오경보",     config.OUTPUTS_DIR / "yawn_fp" / "video_closed_rate.csv"),
    ("YuNet 검출기",           config.YUNET_MODEL),
]
for label, p in checks:
    print(f"  [{'o' if Path(p).exists() else 'X'}] {label:24s} {config._rel(p)}")

### 관찰 결과

- 각 STEP 의 핵심 산출물이 있으면 `[o]`, 없으면 `[X]` 다. `[X]` 인 STEP 은 해당 노트북을 아직 실행하지 않은 것이다.
- 이 노트북은 산출물을 만들지 않는다. 점검만 한다.

## 지금까지의 결과 요약

### 데이터 구축 (10·11)

- DMD 16영상을 프레임 단위 눈 상태 평가 기준으로 변환했으며, mosaic 프레임과의 정렬이 16/16 영상에서 확인되었다.
- 이진 눈 상태 평가 가능 프레임은 74.0%, Closed:Open은 약 1:6.4로 클래스 불균형이 존재했다.
- MRL + DMD를 통합 manifest로 구성했으며, 비대칭 stride를 적용해 합본 train의 Closed 비율을 약 49.6%로 유지했다.
- 데이터 누수 검증 6개 항목은 모두 0이었다.

### 눈 CNN 학습 (12) — DMD hold-out test

| 조건 | Closed-Recall | Precision | F1 |
|---|---:|---:|---:|
| **C (MRL+DMD)** | **0.914** | 0.787 | **0.846** |
| C' (MRL→DMD fine-tuning) | 0.881 | 0.806 | 0.842 |
| A (MRL only) | 0.853 | 0.317 | 0.462 |
| B (DMD only) | 0.597 | 0.955 | 0.735 |

- C(MRL+DMD)가 본 실험의 네 조건 중 **가장 높은 Closed-Recall과 F1**을 기록했다.
- A(MRL only)는 DMD 환경에서 Precision이 낮았고, C에서는 Recall과 Precision이 모두 개선되었다.
- B(DMD only)는 Precision은 높았지만 Closed-Recall이 낮았다.
- 단일 seed와 하나의 hold-out split에 기반한 결과이므로, C가 일반적으로 우수하다고 일반화하지 않는다.

### 파이프라인·검증 (13·14·15)

- **13** 실제 DMD 영상 → YuNet → eye crop → CNN 파이프라인의 Closed-Recall은 **0.916**으로, crop 기반 평가(0.914)와 유사했다. 이번 test에서는 검출 실패가 없었으며, gZ_37에서 0.686으로 낮은 성능이 관찰되었다.
- **14** NITYMED에서는 microsleep과 하품 영상의 길이 차이가 컸으며, 창 개수만으로도 AUC **0.945**가 나타났다. 창 개수의 영향을 고려한 후 세 눈 지표의 AUC는 **0.402~0.424**로 낮아졌다.
- **15** 하품 구간에서 Closed 판정 비율은 **25.7%**, microsleep 샘플 프레임은 **19.9%** 였다. 영상 단위 Closed 비율 AUC는 **0.440 [0.304, 0.577]** 이었다.
- 14·15에서는 눈 감김 지표만으로 microsleep과 하품을 명확하게 구분하는 판별력을 확인하지 못했다.

### 종합 결론

- 눈 CNN은 DMD의 눈 상태 평가에서 Closed-Recall 약 **0.91**, 실제 영상 파이프라인에서 약 **0.92**를 기록했다.
- 반면 NITYMED에서는 **눈 감김 단일 신호만으로 microsleep과 하품을 구분하기 어려운 결과**가 관찰되었다.
- 따라서 후속 위험도 지수에서는 눈 감김 지표와 **하품 등 다른 행동 정보를 함께 고려하는 방안**을 검토할 필요가 있다.

## 한계

- **DMD 조건**: 16개 영상 모두 `Car Stopped`로 실제 주행 환경과 차이가 있다.
- **NITYMED 영상 길이**: microsleep과 하품 영상의 관찰 길이가 달라 영상 단위 지표에 영향을 줄 수 있다.
- **인물 ID 부재**: NITYMED에서 인물 단위 독립성을 완전히 확인하기 어렵다.
- **단일 시드**: STEP12 각 조건을 seed 42로 1회 학습했으므로 C와 C'의 작은 차이는 재현성 검증이 필요하다.
- **임계값 전이**: DMD validation에서 선택한 threshold 0.93을 STEP13~15에 그대로 적용했다.

## 다음 할 일

- **팀원**: 20·21 하품 GT·CNN, 30 위험도 지수 통합
- **눈 트랙**: 19(Colab)을 12로 통합·정리
- 필요시 STEP12를 여러 seed로 반복하여 **C와 C'의 차이 재현성 확인**